# 02 – Làm sạch và chia tập

Đọc 3 nguồn gốc (chỉ đọc), làm sạch, chia Shopee **theo sản phẩm**, tạo nhãn câu cho ViSD4SA, kiểm tra trùng chéo, rồi lưu vào `data/processed/`. Cột `text_prep` là văn bản đã qua `src/preprocess.py` (chuẩn hóa + tách từ), để các notebook sau khỏi chạy lại.

In [1]:
import sys, os, re, json, collections, unicodedata
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, "../src")
from preprocess import preprocess, strip_accents

OUT = "../data/processed"
os.makedirs(OUT, exist_ok=True)
steps = []   # bảng "số dòng sau từng bước làm sạch" cho Chương 3
def log(source, step, n):
    steps.append({"nguồn": source, "bước": step, "số dòng": n})
    print(f"{source:8s} | {step:45s} | {n}")

## 1. Shopee CSV

In [2]:
df = pd.read_csv("../data/final_train_dataset.csv").rename(
    columns={"prod": "product", "com": "text", "rat": "rating"})
log("Shopee", "file gốc", len(df))
df["text"] = df["text"].astype(str).map(lambda s: unicodedata.normalize("NFC", s))
df["product"] = df["product"].map(lambda s: unicodedata.normalize("NFC", s))   # 2 tên đang ở dạng NFD
df["rating"] = df["rating"].astype(int)

# 1) bỏ rác giao diện "Tổng quan\n4.9\n(3034 đánh giá)..."
df = df[~df["text"].str.match(r"^\s*Tổng quan\s*\n\s*\d")]
log("Shopee", "bỏ rác giao diện 'Tổng quan…'", len(df))

# 2) bỏ nhãn mục template "Chất lượng sản phẩm:", "Đúng với mô tả:", ... ở đầu dòng
LABEL_RE = re.compile(r"(?m)^\s*[^:\n]{3,25}:\s*")
df["text_clean"] = (df["text"].str.replace(LABEL_RE, "", regex=True)
                    .str.replace(r"\s+", " ", regex=True).str.strip())

# 3) khử trùng lặp TRƯỚC khi chia, bỏ câu < 2 từ
df = df.drop_duplicates(subset=["text_clean"])
log("Shopee", "khử trùng (sau khi bỏ nhãn mục)", len(df))
df = df[df["text_clean"].str.split().str.len() >= 2]
log("Shopee", "bỏ review < 2 từ", len(df))

# 4) nhãn yếu từ số sao
df["weak_label"] = df["rating"].map({1: "neg", 2: "neg", 3: "neu", 4: "pos", 5: "pos"})
df = df.reset_index(drop=True)
df.insert(0, "id", [f"shp_{i:04d}" for i in range(len(df))])
print(df.weak_label.value_counts())

Shopee   | file gốc                                      | 2843
Shopee   | bỏ rác giao diện 'Tổng quan…'                 | 2723


Shopee   | khử trùng (sau khi bỏ nhãn mục)               | 2556
Shopee   | bỏ review < 2 từ                              | 2548
weak_label
pos    1709
neg     597
neu     242
Name: count, dtype: int64


In [3]:
# Ví dụ trước / sau khi bỏ nhãn mục template
for _, r in df[df.text != df.text_clean].sample(3, random_state=42).iterrows():
    print("TRƯỚC:", r.text[:200].replace("\n", " | "))
    print("SAU  :", r.text_clean[:200]); print()

TRƯỚC: Bao bì/Mẫu mã:tốt | Hương vị:siêu ngon | Bánh ngon hợp khẩu vị lắm   lần sau sẽ tiếp tục ủng hộ nha mình thích nhất là sốt tắc vị chua chua ngọt ngọt ngon lắm các món  topping và sốt ngon nha hợp khẩu vị 
SAU  : tốt siêu ngon Bánh ngon hợp khẩu vị lắm lần sau sẽ tiếp tục ủng hộ nha mình thích nhất là sốt tắc vị chua chua ngọt ngọt ngon lắm các món topping và sốt ngon nha hợp khẩu vị mình hết cho 5 sao nha

TRƯỚC: Màu sắc:be | Chất liệu:conton | Đúng với mô tả:đúng | Sản phẩm đẹp y hình đăng bán đã mua đủ các màu rồi, ôi trời nhà gu áo polo ngon bổ dẻ
SAU  : be conton đúng Sản phẩm đẹp y hình đăng bán đã mua đủ các màu rồi, ôi trời nhà gu áo polo ngon bổ dẻ

TRƯỚC: Chất lượng sản phẩm:tôi rất hài lòng
SAU  : tôi rất hài lòng



In [4]:
# 5) chia theo sản phẩm: 70% sản phẩm để train, 30% làm kho tập vàng
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, gold_idx = next(gss.split(df, groups=df["product"]))
shopee_train, gold_pool = df.iloc[tr_idx].copy(), df.iloc[gold_idx].copy()
assert not set(shopee_train["product"]) & set(gold_pool["product"])   # không sản phẩm nào nằm ở cả hai
print("train:", len(shopee_train), "review,", shopee_train["product"].nunique(), "sản phẩm")
print("gold pool:", len(gold_pool), "review,", gold_pool["product"].nunique(), "sản phẩm")
gold_pool.groupby("product").size().sort_values(ascending=False)

train: 1802 review, 22 sản phẩm
gold pool: 746 review, 10 sản phẩm


product
Nước hoa nam cao cấp chính hãng Jile Guilty 50ml với hương thơm mạnh mẽ, nam tính, lịch lãm, quý phái.    166
Đồng hồ điện tử nam nữ sport S07 kiểu dáng thời trang thể thao                                            116
ÁO BRA DÁN NÂNG NGỰC SILICON CÀI TRƯỚC KHÔNG GỌNG HÌNH BÀN TAY P29 BC_CHIC LINGERIE                        73
Kính Chống Bụi đi đường DOKCRAZY FOGLY cho người đi xe máy, xe đạp siêu Xịn Xò                             73
Kem dưỡng giảm mụn nhanh 8 giờ La Roche-Posay Effaclar Duo+ M 7.5ml                                        63
(Kèm 1 sốt tắc) COMBO Ú Ù (Bánh tráng + Tóp mỡ + túi Bơ + Ruốc hành phi)                                   61
Điện thoại Apple iPhone 16 Pro Max 256GB                                                                   60
Thắt lưng mặt xoay nam cao cấp thời trang, dây lưng da khóa tự động X003, Dây nịt Akedo                    60
[TẶNG MIẾNG NGẪU NHIÊN] Miếng lót sơ sinh Bobby Newborn 1-108 miếng                                        55
Sá

## 2. JSONL không dấu (Aug)

In [5]:
j = pd.read_json("../data/aug_unaccented_reviews.jsonl", lines=True, dtype={"id": str})
log("Aug", "file gốc", len(j))
print("Dòng có *, tab hoặc dấu \":", j["review"].str.contains(r'[*\t"]', regex=True).sum())

# bỏ ký tự gạch đầu dòng "*", tab và dấu ngoặc kép thừa (thay bằng khoảng trắng để câu không dính nhau)
j["review"] = (j["review"].str.replace(r'[*\t"]+', " ", regex=True)
                          .str.replace(r"\s+", " ", regex=True).str.strip())
j = j.drop_duplicates(subset=["review"])
log("Aug", "khử trùng", len(j))
j = j[j["review"].str.split().str.len() >= 2]
log("Aug", "bỏ review < 2 từ", len(j))

keep = ((j.rating <= 2) & (j.label == "negative")) | ((j.rating >= 4) & (j.label == "positive"))
aug = (j[keep].rename(columns={"review": "text"})
       .assign(weak_label=lambda d: d.label.map({"negative": "neg", "positive": "pos"})))
log("Aug", "giữ câu có sao và nhãn khớp (bỏ 3 sao)", len(aug))
print(aug.weak_label.value_counts())

Aug      | file gốc                                      | 1347
Dòng có *, tab hoặc dấu ": 55


Aug      | khử trùng                                     | 1347
Aug      | bỏ review < 2 từ                              | 1347


Aug      | giữ câu có sao và nhãn khớp (bỏ 3 sao)        | 1205
weak_label
pos    652
neg    553
Name: count, dtype: int64


## 3. ViSD4SA: nhãn cụm → nhãn câu + 3 cột khía cạnh

Quy tắc: mọi cụm cùng cảm xúc → lấy cảm xúc đó; lẫn lộn → lấy cảm xúc của `GENERAL` nếu có, không thì lấy đa số; hòa → **bỏ câu** (không gán neu).

In [6]:
MAP4 = {"PRICE": "gia_ca", "SER&ACC": "dich_vu"}          # 8 khía cạnh còn lại -> chat_luong
POL = {"POSITIVE": "pos", "NEGATIVE": "neg", "NEUTRAL": "neu"}

def sentence_label(spans):
    pols = [t.split("#")[1] for _, _, t in spans]
    if not pols:
        return None
    if len(set(pols)) == 1:
        return POL[pols[0]]
    gen = [t.split("#")[1] for _, _, t in spans if t.startswith("GENERAL#")]
    if gen:
        return POL[collections.Counter(gen).most_common(1)[0][0]]
    (p1, c1), (p2, c2) = collections.Counter(pols).most_common(2)
    return None if c1 == c2 else POL[p1]

def load_visd(split):
    rows = []
    for line in open(f"../ViSD4SA/{split}.jsonl", encoding="utf-8"):
        r = json.loads(line)
        text = r["text"]
        aspects = {}
        for st, en, tag in r["labels"]:
            en = min(en, len(text))                        # sửa 122 cụm có end vượt độ dài câu
            asp, pol = tag.split("#")
            aspects.setdefault(MAP4.get(asp, "chat_luong"), []).append(POL[pol])
        rows.append({"text": unicodedata.normalize("NFC", text),   # NFC SAU khi đã dùng offset
                     "label": sentence_label(r["labels"]),
                     "n_spans": len(r["labels"]),
                     **{a: collections.Counter(v).most_common(1)[0][0] for a, v in aspects.items()}})
    return pd.DataFrame(rows)

visd = {}
for sp in ["train", "dev", "test"]:
    d = load_visd(sp)
    log(f"A-{sp}", "file gốc", len(d))
    d = d[d.n_spans > 0]
    log(f"A-{sp}", "bỏ review không có cụm nào", len(d))
    d = d.dropna(subset=["label"])
    log(f"A-{sp}", "bỏ review hòa cảm xúc", len(d))
    visd[sp] = d.drop(columns="n_spans").reset_index(drop=True)

allv = pd.concat(visd.values())
print(allv.label.value_counts(), allv.label.value_counts(normalize=True).round(3), sep="\n")

A-train  | file gốc                                      | 7785
A-train  | bỏ review không có cụm nào                    | 7625
A-train  | bỏ review hòa cảm xúc                         | 7347
A-dev    | file gốc                                      | 1112
A-dev    | bỏ review không có cụm nào                    | 1086
A-dev    | bỏ review hòa cảm xúc                         | 1046


A-test   | file gốc                                      | 2225
A-test   | bỏ review không có cụm nào                    | 2186
A-test   | bỏ review hòa cảm xúc                         | 2120
label
pos    6638
neg    3456
neu     419
Name: count, dtype: int64
label
pos    0.631
neg    0.329
neu    0.040
Name: proportion, dtype: float64


In [7]:
# Xóa câu trùng giữa train và dev/test của ViSD4SA (xóa khỏi train)
n0 = len(visd["train"])
visd["train"] = visd["train"][~visd["train"].text.isin(set(visd["dev"].text) | set(visd["test"].text))]
log("A-train", "bỏ câu trùng với dev/test", len(visd["train"]))

A-train  | bỏ câu trùng với dev/test                     | 7344


## 4. Kiểm tra trùng chéo với kho tập vàng

Chuẩn hóa mạnh (bỏ dấu, viết thường, bỏ dấu câu) rồi so. Câu nào của tập train trùng với kho tập vàng thì **xóa khỏi train**.

In [8]:
def key(s):
    s = strip_accents(unicodedata.normalize("NFC", str(s))).lower()
    return re.sub(r"[^a-z0-9]+", "", s)

gold_keys = set(gold_pool.text_clean.map(key))
for name, d, col in [("Shopee-train", shopee_train, "text_clean"), ("Aug", aug, "text"),
                     ("A-train", visd["train"], "text"), ("A-dev", visd["dev"], "text"), ("A-test", visd["test"], "text")]:
    print(f"{name:13s} trùng với kho tập vàng: {d[col].map(key).isin(gold_keys).sum()}")

shopee_train = shopee_train[~shopee_train.text_clean.map(key).isin(gold_keys)]
aug = aug[~aug.text.map(key).isin(gold_keys)]
visd["train"] = visd["train"][~visd["train"].text.map(key).isin(gold_keys)]
log("Shopee", "train sau khi bỏ trùng với kho vàng", len(shopee_train))
log("Aug", "sau khi bỏ trùng với kho vàng", len(aug))
log("A-train", "sau khi bỏ trùng với kho vàng", len(visd["train"]))

Shopee-train  trùng với kho tập vàng: 8


Aug           trùng với kho tập vàng: 0


A-train       trùng với kho tập vàng: 0
A-dev         trùng với kho tập vàng: 0


A-test        trùng với kho tập vàng: 0


Shopee   | train sau khi bỏ trùng với kho vàng           | 1794
Aug      | sau khi bỏ trùng với kho vàng                 | 1205
A-train  | sau khi bỏ trùng với kho vàng                 | 7344


## 5. Tiền xử lý (chuẩn hóa + tách từ) và lưu file

Tách từ bằng underthesea mất khoảng 1–2 phút.

In [9]:
%%time
shopee_train["text_prep"] = shopee_train.text_clean.map(preprocess)
gold_pool["text_prep"] = gold_pool.text_clean.map(preprocess)
aug["text_prep"] = aug.text.map(preprocess)
for sp in visd:
    visd[sp]["text_prep"] = visd[sp].text.map(preprocess)

CPU times: total: 1min 33s
Wall time: 1min 35s


In [10]:
cols_shp = ["id", "product", "rating", "text", "text_clean", "text_prep", "weak_label"]
shopee_train[cols_shp].to_csv(f"{OUT}/shopee_train.csv", index=False)
gold_pool[cols_shp].to_csv(f"{OUT}/shopee_gold_pool.csv", index=False)
df[cols_shp[:-2] + ["weak_label"]].to_csv(f"{OUT}/shopee_clean_all.csv", index=False)   # toàn bộ 2.5k review cho phần kinh doanh
aug[["id", "rating", "label", "text", "text_prep", "weak_label"]].to_csv(f"{OUT}/aug_train.csv", index=False)
for sp, d in visd.items():
    cols = ["text", "text_prep", "label"] + [c for c in ["chat_luong", "gia_ca", "dich_vu"] if c in d]
    d[cols].to_csv(f"{OUT}/visd_{sp}.csv", index=False)

steps_df = pd.DataFrame(steps)
steps_df.to_csv(f"{OUT}/cleaning_steps.csv", index=False)
print(sorted(os.listdir(OUT)))
steps_df

['aug_train.csv', 'cleaning_steps.csv', 'product_category.csv', 'shopee_clean_all.csv', 'shopee_gold_pool.csv', 'shopee_train.csv', 'visd_dev.csv', 'visd_test.csv', 'visd_train.csv']


,nguồn,bước,số dòng
0,Shopee,file gốc,2843
1,Shopee,bỏ rác giao diện 'Tổng quan…',2723
2,Shopee,khử trùng (sau khi bỏ nhãn mục),2556
3,Shopee,bỏ review < 2 từ,2548
4,Aug,file gốc,1347
5,Aug,khử trùng,1347
6,Aug,bỏ review < 2 từ,1347
7,Aug,giữ câu có sao và nhãn khớp (bỏ 3 sao),1205
8,A-train,file gốc,7785
9,A-train,bỏ review không có cụm nào,7625


In [11]:
# Ví dụ trước / sau tiền xử lý (cho Chương 4)
ex = shopee_train.sample(5, random_state=1)
for _, r in ex.iterrows():
    print("GỐC :", r.text_clean[:160]); print("PREP:", r.text_prep[:160]); print()

GỐC : Mùi khá ổn, dịu nhẹ, sau khi rửa không làm khô da.
PREP: mùi khá ổn , dịu nhẹ , sau khi rửa không làm khô da .

GỐC : tuyệt vời đúng hình Giao hàng siêu nhanh ,hàng rẻ mà đẹp bé nhà mình rất thích về chơi suốt đi ngủ cũng ôm nữa chơi 2 ngày r mà k chán,biết vậy mlaays cái sạc đ
PREP: tuyệt_vời đúng hình giao hàng siêu nhanh , hàng rẻ mà đẹp bé nhà mình rất thích về chơi suốt đi ngủ cũng ôm nữa chơi 2 ngày rồi mà không chán , biết vậy mlaays 

GỐC : Mua dc giảm giá. Chưa dùng nên chưa biết
PREP: mua được giảm_giá . chưa dùng nên chưa biết

GỐC : Chất lượng sản phẩm tốt Shop giao hàng siêu nhanh Đóng gói cẩn thận
PREP: chất_lượng sản_phẩm tốt cửa_hàng giao hàng siêu nhanh đóng_gói cẩn_thận

GỐC : đúng ok Mua shop lần thứ N rùi nhưng lần này sale đỉnh qá bình thường mình mua 1 cái cũng 500 🐟 mà lần này sale còn 325 🐟 nón xinh, rất chắc và nặng tay.
PREP: đúng ổn mua cửa_hàng lần thứ n rồi nhưng lần này sale đỉnh quá bình_thường mình mua 1 cái cũng 500 EMO_fish mà lần này sale còn 32